# 01.08 Cohere

Cohere Command R+ 계열은 RAG·검색 품질을 중시하는 엔터프라이즈 워크로드에 맞게 설계됐다. Cohere 고유 기능으로 **`documents` 파라미터**(RAG 시그널 주입)와 **`citations`** 출력이 있다.

## 학습 목표

- `ChatCohere(model="command-r-plus")` 기본 사용
- `cohere_api_key` 인자 이름(주의: `api_key` 아님) 매핑
- tool calling / structured output

## 언제 쓰나

- Cohere Rerank / Embed 와 같이 쓰는 RAG 파이프라인
- Command A / R+ 의 다국어 성능이 필요할 때

## 01.08.1 환경 설정

In [ ]:
# !pip install -U langchain langchain-cohere
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("COHERE_API_KEY"), "COHERE_API_KEY 필요"

## 01.08.2 기본 사용

`ChatCohere.__init__`은 일반 `api_key` 대신 **`cohere_api_key`** 필드를 쓴다. 환경변수 `COHERE_API_KEY`는 자동 매핑된다.

In [ ]:
from langchain_cohere import ChatCohere

model = ChatCohere(model="command-r-plus", temperature=0)
msg = model.invoke("Cohere Command 계열의 강점을 한 문장으로 설명해줘.")
print(msg.content)
print("usage:", msg.usage_metadata)

## 01.08.3 `init_chat_model()` 경로

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model("cohere:command-r")
print(m.invoke("핑").content[:80])

## 01.08.4 Tool calling

In [ ]:
from langchain_core.tools import tool

@tool
def country(code: str) -> str:
    """ISO 국가 코드를 국가명으로 변환한다."""
    return {"KR": "한국", "JP": "일본", "US": "미국"}.get(code.upper(), "?")

bound = model.bind_tools([country])
out = bound.invoke("KR 은 어느 나라지?")
print("tool_calls:", out.tool_calls)

## 01.08.5 Structured output

In [ ]:
from pydantic import BaseModel

class Country(BaseModel):
    code: str
    name: str

structured = model.with_structured_output(Country)
print(structured.invoke("JP 는 일본이야. 구조화해줘."))

## 01.08.6 Cohere-native RAG — `documents` 파라미터

Cohere의 chat API는 `documents=[{"title":..., "snippet":...}, ...]` 를 받아 내부적으로 citation을 생성한다. LangChain 래퍼는 이를 `invoke(..., documents=[...])` 로 노출한다. (간단한 샘플 — 실제 RAG 에서는 vector store 결과를 snippet으로 넣는다.)

In [ ]:
docs = [
    {"title": "LangChain", "snippet": "LangChain은 LLM 애플리케이션 프레임워크다."},
    {"title": "Cohere", "snippet": "Cohere는 엔터프라이즈 LLM 제공자다."},
]
resp = model.invoke("LangChain이 뭐야?", documents=docs)
print("content     :", resp.content[:200])
print("citations   :", (resp.response_metadata or {}).get("citations"))

## 체크리스트

- [ ] 인자 이름이 `cohere_api_key`(환경변수 매핑)라는 점을 기억했다
- [ ] `command-r` / `command-r-plus` / `command-a` 모델 계층을 이해했다
- [ ] `documents=` 로 RAG 시그널을 넘기면 `citations` 메타가 돌아온다

## 다음

- `02_embeddings/03_cohere.ipynb` — Cohere Embed 조합
- `09_routers_and_aggregators.ipynb` — 멀티-프로바이더 래핑

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/cohere